In [1]:
# ============================================================
# Import Required Libraries
# ============================================================

import os
import json
import torch

from tqdm import tqdm

from PIL import Image

from transformers import (
    Qwen2_5_VLForConditionalGeneration,
    AutoProcessor,
    BitsAndBytesConfig
)

from peft import (
    PeftModel,
    LoraConfig,
    get_peft_model,
    TaskType
)

print("=" * 60)
print("Libraries Imported Successfully")
print("=" * 60)

Libraries Imported Successfully


In [2]:
# ============================================================
# GPU Configuration
# ============================================================

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("=" * 60)
print("Device :", device)

if torch.cuda.is_available():
    print("GPU :", torch.cuda.get_device_name(0))

print("=" * 60)

Device : cuda
GPU : NVIDIA GeForce RTX 3050 6GB Laptop GPU


In [3]:
# ============================================================
# Load Student Model
# ============================================================

MODEL_NAME = "Qwen/Qwen2.5-VL-3B-Instruct"

bnb_config = BitsAndBytesConfig(

    load_in_4bit=True,

    bnb_4bit_quant_type="nf4",

    bnb_4bit_compute_dtype=torch.float16,

    bnb_4bit_use_double_quant=True

)

processor = AutoProcessor.from_pretrained(MODEL_NAME)

student_model = Qwen2_5_VLForConditionalGeneration.from_pretrained(

    MODEL_NAME,

    quantization_config=bnb_config,

    torch_dtype=torch.float16,

    device_map="auto"

)

print("=" * 60)
print("Student Model Loaded")
print("=" * 60)

W0728 15:22:07.962000 26864 site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels


Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/824 [00:00<?, ?it/s]

Student Model Loaded


In [4]:
# ============================================================
# Apply LoRA
# ============================================================

lora_config = LoraConfig(

    r=16,

    lora_alpha=32,

    lora_dropout=0.05,

    bias="none",

    task_type=TaskType.CAUSAL_LM,

    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj"
    ]

)

student_model = get_peft_model(
    student_model,
    lora_config
)

student_model.eval()

print("=" * 60)
print("LoRA Loaded")
print("=" * 60)

LoRA Loaded


In [19]:
# ============================================================
# Dataset Configuration
# ============================================================

import os
from PIL import Image
from torch.utils.data import Dataset
from torchvision import transforms

DATASET_PATH = "processed_dataset"

TRAIN_IMAGES = os.path.join(DATASET_PATH, "train", "images")
TRAIN_REPORTS = os.path.join(DATASET_PATH, "train", "reports")

transform = None

In [20]:
# ============================================================
# Dataset Class
# ============================================================

class ChestXrayDataset(Dataset):

    def __init__(self, image_dir, report_dir, transform=None):

        self.image_dir = image_dir
        self.report_dir = report_dir
        self.transform = transform

        self.images = sorted(os.listdir(image_dir))

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):

        image_name = self.images[idx]

        image_path = os.path.join(self.image_dir, image_name)

        report_name = os.path.splitext(image_name)[0] + ".txt"

        report_path = os.path.join(self.report_dir, report_name)

        # Load Original PIL Image (Do NOT Resize)
        image = Image.open(image_path).convert("RGB")

        # Read Ground Truth Report
        with open(report_path, "r", encoding="utf-8") as f:
            report = f.read()

        return {
            "image": image,
            "report": report,
            "image_path": image_path,
            "report_path": report_path
        }

In [21]:
# ============================================================
# Create Dataset
# ============================================================

train_dataset = ChestXrayDataset(

    image_dir=TRAIN_IMAGES,

    report_dir=TRAIN_REPORTS,

    transform=None

)

print("=" * 60)
print("Training Samples :", len(train_dataset))
print("=" * 60)

Training Samples : 21443


In [12]:
# ============================================================
# Load One Chest X-ray Sample
# ============================================================

sample = train_dataset[0]

image = sample["image"]

report = sample["report"]

print("="*60)
print("Ground Truth Report")
print("="*60)

print(report[:700])

Ground Truth Report
The lungs are clear of focal consolidation, pleural effusion or pneumothorax. The heart size is normal. The mediastinal contours are normal. Multiple surgical clips project over the left breast, and old left rib fractures are noted.  No acute cardiopulmonary process.


In [22]:
# ============================================================
# Medical Prompt
# ============================================================

prompt = """
You are a board-certified radiologist.

A chest X-ray image is attached.

Your task is to carefully inspect the image and generate a complete radiology interpretation.

Do NOT say that you cannot see or access the image.

Think carefully before answering.

Return your answer in exactly this format.

Findings:
- ...

Reasoning:
- ...

Final Diagnosis:
- ...
"""

In [14]:
# ============================================================
# Prepare Multimodal Input
# ============================================================

from qwen_vl_utils import process_vision_info

messages = [
    {
        "role": "user",
        "content": [
            {
                "type": "image",
                "image": image
            },
            {
                "type": "text",
                "text": prompt
            }
        ]
    }
]

text = processor.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

image_inputs, video_inputs = process_vision_info(messages)

inputs = processor(
    text=[text],
    images=image_inputs,
    videos=video_inputs,
    padding=True,
    return_tensors="pt"
)

inputs = inputs.to(device)

print("="*60)
print("Input Prepared Successfully")
print("="*60)

Input Prepared Successfully


In [23]:
# ============================================================
# Generate Reasoning Trajectories
# ============================================================

NUM_TRAJECTORIES = 5

trajectories = []

student_model.eval()

with torch.no_grad():

    for i in range(NUM_TRAJECTORIES):

        generated_ids = student_model.generate(

            **inputs,

            max_new_tokens=256,

            do_sample=False

        )

        generated_ids = generated_ids[:, inputs.input_ids.shape[1]:]

        response = processor.batch_decode(

            generated_ids,

            skip_special_tokens=True,

            clean_up_tokenization_spaces=False

        )[0]

        trajectories.append(response)

        print(f"Trajectory {i+1} Generated")

print("=" * 60)
print("All Trajectories Generated Successfully")
print("=" * 60)

Trajectory 1 Generated
Trajectory 2 Generated
Trajectory 3 Generated
Trajectory 4 Generated
Trajectory 5 Generated
All Trajectories Generated Successfully


In [24]:
# ============================================================
# Display Results
# ============================================================

for i, trajectory in enumerate(trajectories):

    print("="*80)
    print(f"Trajectory {i+1}")
    print("="*80)

    print(trajectory)

    print()

Trajectory 1
I'm sorry, but I can't view or analyze images directly. However, I can guide you on how to interpret a chest X-ray yourself if you have the image available. If you need help with interpreting the findings, please describe the X-ray and any symptoms you're experiencing, and I'll provide guidance based on common patterns seen in chest X-rays.

Trajectory 2
I'm sorry, but I can't view or analyze images directly. However, I can guide you on how to interpret a chest X-ray yourself if you have the image available. If you need help with interpreting the findings, please describe the X-ray and any symptoms you're experiencing, and I'll provide guidance based on common patterns seen in chest X-rays.

Trajectory 3
I'm sorry, but I can't view or analyze images directly. However, I can guide you on how to interpret a chest X-ray yourself if you have the image available. If you need help with interpreting the findings, please describe the X-ray and any symptoms you're experiencing, and

In [17]:
# # ============================================================
# # Save Generated Trajectories
# # ============================================================

# import json
# import os

# os.makedirs("generated_trajectories", exist_ok=True)

# result = {

#     "ground_truth": report,

#     "trajectories": trajectories

# }

# with open(

#     "generated_trajectories/sample_000001.json",

#     "w",

#     encoding="utf-8"

# ) as f:

#     json.dump(result, f, indent=4)

# print("="*60)
# print("Trajectories Saved Successfully")
# print("="*60)

Trajectories Saved Successfully


In [18]:
import transformers
import qwen_vl_utils

print(transformers.__version__)
print(qwen_vl_utils.__version__)

5.14.1


AttributeError: module 'qwen_vl_utils' has no attribute '__version__'